# Random Forest — multibranch-style sequence classifier

This notebook uses the cleaned `base_utils_qwen.py` and trains a sequence-level Random Forest.

Local quick runs use `data/sample.csv` (37 sequences). Set `use_sample_data = False` for full `train.csv`.

Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

Style:
- configuration
- data loading
- split
- estimator
- parameter search (grid or Bayesian)
- holdout evaluation
- save results


In [14]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer
from skopt.space import Real, Integer, Categorical
from sklearn.pipeline import Pipeline

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from src/')
except ImportError:
    import data_utils
    from base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from flat src path')


Imports loaded from src/


In [15]:
# Install optional search / feature dependencies if missing (safe to re-run)
for package_name, import_name in [
    ('scikit-optimize', 'skopt'),
    ('PyWavelets', 'pywt'),
]:
    try:
        __import__(import_name)
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        print(f'Installed {package_name}')

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print(f'scikit-optimize {skopt.__version__} ready')
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False
    print('scikit-optimize unavailable')

try:
    import pywt
    print(f'PyWavelets {pywt.__version__} ready')
except ImportError:
    print('PyWavelets unavailable')

scikit-optimize 0.10.2 ready
PyWavelets 1.8.0 ready


In [ ]:
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'  # also available: eg.csv

search_mode = 'grid'  # 'grid' or 'bayesian' for quick smoke test use 'grid'
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.4
train_size = 0.6
n_iter = 10            # Bayesian iterations for a quick smoke test
verbose = 3
error_score = np.nan  # 'raise' or 'warn'

results_dir = Path('results_rf_multibranch_style')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)


In [17]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_df = pd.read_csv(data_root / 'train_demographics.csv')

train_df = raw_train_df.set_index('row_id').copy(deep=True)

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

problematic_sequence_df = train_df.groupby('sequence_id')[['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z']].skew().abs()
ideal_skew_threshold = 1.6
problematic_features_threshold = 3
result_series = ((problematic_sequence_df > ideal_skew_threshold).sum(axis=1) >= problematic_features_threshold)
problematic_sequences_list = result_series.loc[result_series].index
train_df['problematic_sequence'] = train_df['sequence_id'].isin(problematic_sequences_list).astype(bool)

if TARGET_COL not in train_df.columns:
    train_df[TARGET_COL] = train_df['gesture_action']

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)


Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Using train.csv: 8151 sequences


In [18]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception:
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(seq_df, groups=seq_df['sequence_id']))

    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())


Train: 2910 seqs | 35.7%
Test:  969 seqs  | 11.9%
Train sequences: 2910
Test sequences: 969


In [19]:
rf_pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col="sequence_id",
        counter_col="sequence_counter",
    )),
    ('extractor', SequenceExtractor(
        acc_modes='raw|velocity|jerk',
        rotation_modes='quaternion|angular_velocity',
    )),
    ('estimator', RandomForestClassifier(
        n_estimators=300,
        random_state=random_state,
    ))
])

In [ ]:
# ============================================================
# GRID SEARCH SPACE
# Practical compact grid. Bayesian space does the full exploration.
# ============================================================

GRID_PARAM_SPACE = {
    # ------------------------------------------------------------
    # EXTRACTOR: main sensor-feature domains
    # ------------------------------------------------------------
    'extractor__acc_modes': [
        'smoothed|velocity|displacement|jerk',
        None
    ],
    'extractor__rotation_modes': [
        'quaternion|euler|angular_velocity',
        None
    ],
    'extractor__tof_modes': [
        'pooled_stats|sensor_stats',
        None
    ],
    'extractor__thm_modes': [
        'centered_diff',
    ],
    'extractor__frame_stats': [
        'mean,std,min,max,last,first,rms',
        None
    ],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed preprocessing for stable RF smoke/grid runs
    # ------------------------------------------------------------
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],

    'extractor__window_size': [20],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed frame-output safety params
    # ------------------------------------------------------------
    'extractor__output_format': ['frame'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160],
    'extractor__chunk_window_size': [50],
    'extractor__chunk_stride': [25],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],

    'extractor__imu_native_sampling_rate': [100],
    'extractor__rot_native_sampling_rate': [100],
    'extractor__tof_native_sampling_rate': [20],
    'extractor__thm_native_sampling_rate': [20],

    'extractor__imu_target_sampling_rate': [100],
    'extractor__rot_target_sampling_rate': [100],
    'extractor__tof_target_sampling_rate': [20],
    'extractor__thm_target_sampling_rate': [20],

    # ------------------------------------------------------------
    # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - FIXED FOR GRID
    # ------------------------------------------------------------
    'extractor__stft_nperseg': [None, 100],
    'extractor__stft_noverlap': [50],
    'extractor__stft_window_type': ['hann'],
    'extractor__stft_use_log_scale': [True],

    'extractor__cwt_wavelet': ['morl'],
    'extractor__cwt_max_scale': [32],
    'extractor__cwt_n_scales': [32],
    'extractor__cwt_use_log_scale': [False],

    # ------------------------------------------------------------
    # AUGMENTOR: Baseline (Set to 0.0 to switch off)
    # Uses EXACT valid parameter names from your SensorAugmentor class
    # ------------------------------------------------------------
    'augmentor__prob': [0.0],                 # Overall probability of augmenting a sequence
    'augmentor__per_aug_prob': [0.0],         # Probability of applying each specific augmentation
    'augmentor__jitter_sigma': [0.0],
    'augmentor__noise_std': [0.0],            # Replaces gaussian_snr_db
    'augmentor__scaling_sigma': [0.0],        # Replaces scaling_range
    'augmentor__sensor_drop_prob': [0.0],     # Replaces sensor_dropout_prob
    'augmentor__channel_drop_prob': [0.0],    # Replaces channel_dropout_prob/rate
    'augmentor__time_shift_frac': [0.0],      # Replaces timeshift_max_frac
    'augmentor__crop_frac_range': [(0.5, 1.0)], # Replaces crop_min_frac
    'augmentor__temporal_mask_frac': [0.0],
    'augmentor__temporal_num_masks': [0],
    'augmentor__warp_sigma': [0.0],           # Replaces magwarp_sigma
    'augmentor__warp_num_knots': [4],         # Replaces magwarp_knots

    # ------------------------------------------------------------
    # RANDOM FOREST ESTIMATOR
    # ------------------------------------------------------------
    'estimator__n_estimators': [100],
    'estimator__criterion': ['gini'],
    'estimator__max_depth': [30],
    'estimator__min_samples_split': [15],
    'estimator__min_samples_leaf': [10],
    'estimator__max_features': ['sqrt'],
    'estimator__bootstrap': [True],
    'estimator__class_weight': ['balanced'],
}


# ============================================================
# BAYESIAN SEARCH SPACE
# Full exploration space.
# ============================================================

if SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            # --------------------------------------------------------
            # EXTRACTOR: sensor-feature domains
            # --------------------------------------------------------
            'extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'smoothed|velocity|displacement|jerk',
                None
            ]),
            'extractor__rotation_modes': Categorical([
                'quaternion|euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler|rot6d',
                None
            ]),
            'extractor__tof_modes': Categorical([
                'pooled_stats|sensor_stats',
                None
            ]),
            'extractor__thm_modes': Categorical([
                'centered_diff',
            ]),
            'extractor__frame_stats': Categorical([
                'mean,std,min,max,last',
                'mean,std,min,max,last,first,rms,abs_mean',
                None
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: preprocessing / filtering
            # --------------------------------------------------------
            'extractor__motion_filter_mode': Categorical([
                'extended_kalman',
            ]),
            'extractor__use_dead_reckoning': Categorical([True]),
            'extractor__dead_reckoning_detrend': Categorical([True]),
            
            'extractor__kalman_process_noise': Real(1e-5, 1e-1, prior='log-uniform'),
            'extractor__kalman_measurement_noise': Real(1e-3, 1e1, prior='log-uniform'),
            
            'extractor__window_size': Categorical([5]),
            'extractor__smooth_alpha': Categorical([None, 0.20, 0.90]),
            'extractor__clip_value': Categorical([150.0]),
            'extractor__interp_mode': Categorical(['linear']),

            # --------------------------------------------------------
            # EXTRACTOR: STFT & CWT (Time-Frequency Domains) - BAYESIAN
            # --------------------------------------------------------
            'extractor__stft_nperseg': Categorical([32, 64, 128]),
            'extractor__stft_noverlap': Categorical([8, 16, 32]),
            'extractor__stft_window_type': Categorical(['hann', 'hamming', 'blackman']),
            'extractor__stft_use_log_scale': Categorical([True, False]),

            'extractor__cwt_wavelet': Categorical(['morl', 'mexh', 'gaus1', 'gaus2']),
            'extractor__cwt_max_scale': Categorical([64, 128, 256]),
            'extractor__cwt_n_scales': Categorical([16, 32, 64]),
            'extractor__cwt_use_log_scale': Categorical([True, False]),

            # --------------------------------------------------------
            # EXTRACTOR: fixed frame-output safety params
            # --------------------------------------------------------
            'extractor__output_format': Categorical(['frame']),
            'extractor__padding_value': Categorical([0.0]),
            'extractor__maxlen': Categorical([150]),
            'extractor__chunk_window_size': Categorical([100]),
            'extractor__chunk_stride': Categorical([50]),
            'extractor__add_global_context': Categorical([True]),
            'extractor__compute_dt': Categorical([True]),

            'extractor__imu_native_sampling_rate': Categorical([100]),
            'extractor__rot_native_sampling_rate': Categorical([100]),
            'extractor__tof_native_sampling_rate': Categorical([20]),
            'extractor__thm_native_sampling_rate': Categorical([20]),

            'extractor__imu_target_sampling_rate': Categorical([100]),
            'extractor__rot_target_sampling_rate': Categorical([100]),
            'extractor__tof_target_sampling_rate': Categorical([20]),
            'extractor__thm_target_sampling_rate': Categorical([20]),
            'extractor__resample_modalities': Categorical([True]),

            # --------------------------------------------------------
            # AUGMENTOR: Full Exploration
            # Uses EXACT valid parameter names from your SensorAugmentor class
            # --------------------------------------------------------
            'augmentor__prob': Real(0.0, 0.6),
            'augmentor__per_aug_prob': Real(0.0, 0.6),
            'augmentor__jitter_sigma': Real(0.0, 0.1),
            'augmentor__noise_std': Real(0.0, 0.1),
            'augmentor__scaling_sigma': Real(0.0, 0.2),
            'augmentor__sensor_drop_prob': Real(0.0, 0.3),
            'augmentor__channel_drop_prob': Real(0.0, 0.3),
            'augmentor__time_shift_frac': Real(0.0, 0.2),
            'augmentor__crop_frac_range': Categorical([(0.5, 1.0), (0.7, 1.0), (0.8, 1.0)]),
            'augmentor__temporal_mask_frac': Real(0.0, 0.25),
            'augmentor__temporal_num_masks': Integer(1, 3),
            'augmentor__warp_sigma': Real(0.0, 0.3),
            'augmentor__warp_num_knots': Integer(2, 6),

            # --------------------------------------------------------
            # RANDOM FOREST ESTIMATOR
            # --------------------------------------------------------
            'estimator__n_estimators': Integer(5, 300),
            'estimator__criterion': Categorical(['gini', 'entropy']),
            'estimator__max_depth': Categorical([None, 10, 30, 50, 100]),
            'estimator__min_samples_split': Integer(2, 20),
            'estimator__min_samples_leaf': Integer(1, 8),
            'estimator__max_features': Categorical([0.3, 0.5, 0.7]),
            'estimator__bootstrap': Categorical([True]),
            'estimator__class_weight': Categorical(['balanced']),
            'estimator__min_impurity_decrease': Real(0.0, 0.005),
        }

    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE

In [42]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=rf_pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)


Fitting 1 folds for each of 64 candidates, totalling 64 fits
[CV 1/1] END augmentor__channel_drop_prob=0.0, augmentor__crop_frac_range=(0.5, 1.0), augmentor__jitter_sigma=0.0, augmentor__noise_std=0.0, augmentor__per_aug_prob=0.0, augmentor__prob=0.0, augmentor__scaling_sigma=0.0, augmentor__sensor_drop_prob=0.0, augmentor__temporal_mask_frac=0.0, augmentor__temporal_num_masks=0, augmentor__time_shift_frac=0.0, augmentor__warp_num_knots=4, augmentor__warp_sigma=0.0, estimator__bootstrap=True, estimator__class_weight=None, estimator__criterion=gini, estimator__max_depth=20, estimator__max_features=sqrt, estimator__min_samples_leaf=5, estimator__min_samples_split=10, estimator__n_estimators=37, extractor__acc_modes=smoothed|velocity|displacement|jerk, extractor__add_global_context=False, extractor__chunk_stride=25, extractor__chunk_window_size=50, extractor__clip_value=None, extractor__compute_dt=True, extractor__cwt_max_scale=32, extractor__cwt_n_scales=32, extractor__cwt_use_log_scale=

KeyboardInterrupt: 

In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'rf_multibranch_style_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'rf_multibranch_style_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'rf_multibranch_style_best_{timestamp}.csv',
    index=False,
)


In [ ]:
importances = pd.Series(
    best_model.estimator_.feature_importances_,
    index=best_model.extractor_.frame_feature_names_,
).sort_values(ascending=False)

print(importances.head(50))
